# Pixel ridge · 01 可调整的噪声曲线与选参比较

默认使用 Van Hateren **完整 10000 维**协方差谱与 disk teacher。回归直接在 input pixel space（用 PC 坐标计算），不投影、不 whitening。

三列：固定 λ；DE 近似 LOOCV 按 E_gen 选 alpha；按 E_acc 选 alpha 的 teacher-aware oracle。每个噪声都重新选参，图同时显示 sigma 和 σ²/S 两种横轴。

首次 Run All 计算并缓存；之后重画只读取 CSV。实际逐 trial RidgeCV 与 MC 在 notebook 02 中。DE_gen_CV 不是已执行的 empirical RidgeCV。


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, time
os.environ.setdefault("MPLCONFIGDIR", "/tmp/accentuationpredrmt-matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/accentuationpredrmt-xdg-cache")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"rmt_core").is_dir())
sys.path.insert(0, str(ROOT))
from scripts import pixel_ridge_notebook_utils as u
threads = threadpool_limits(limits=2)
OUT = ROOT/"notebooks/outputs/pixel_ridge"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# vanhateren / ffhq: full measured input spectrum, no top-PC truncation.
# powerlaw / isotropic: synthetic pixel covariance; D controls dimension.
DATASET = "vanhateren"
D, N = 128, 1000
SEED = 42
RATIOS = np.r_[0., np.geomspace(1e-6, 10., 25)]
ALPHAS = np.logspace(-5, 7, 181)  # alpha = n * lambda
FIXED_LAMBDA = 0.1
ACC_OBJECTIVE = "E_acc"          # or "E_acc_corrected"
MAX_SECONDS = 180
FORCE = False
s, beta = u.load_problem(ROOT, DATASET, D, SEED)
S = float(s @ beta**2)
print("d =", len(s), "n =", N, "signal variance =", S)


## 搜索与缓存

在整个 log(alpha) 网格扫描，再细化每个检测到的局部极小点，比较端点与所有候选点。结果是给定区间内的数值最优；boundary=True 时扩大 ALPHAS 范围再验证。GEN 使用 n−1 风险选 alpha，n 样本 refit；ACC 使用 n 样本的所选理论目标。

In [ ]:
config = dict(dataset=DATASET, d=len(s), n=N, seed=SEED,
              ratios=RATIOS.tolist(), alphas=ALPHAS.tolist(),
              fixed_lambda=FIXED_LAMBDA, acc_objective=ACC_OBJECTIVE)
digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode())
digest.update(s.tobytes()); digest.update(beta.tobytes())
for path in [Path(u.__file__), *sorted((ROOT/"rmt_core").glob("*.py"))]:
    digest.update(path.read_bytes())
run_dir = OUT/digest.hexdigest()[:16]
run_dir.mkdir(parents=True, exist_ok=True)
LOG = run_dir/"progress.log"
def log(message):
    print(message, flush=True)
    with LOG.open("a") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S") + " | " + message + "\n")
log("Cache/log directory: " + str(run_dir))
if (run_dir/"summary.csv").exists() and not FORCE:
    summary = pd.read_csv(run_dir/"summary.csv")
    paths = pd.read_csv(run_dir/"paths.csv")
    log("Loaded plot-ready tables.")
else:
    start = time.perf_counter()
    u.sweep(s, beta, N, RATIOS[-1:], ALPHAS, FIXED_LAMBDA, ACC_OBJECTIVE, progress=False)
    eta = (time.perf_counter()-start)*len(RATIOS)
    log(f"One-noise pilot ETA: {eta:.1f}s")
    if eta > MAX_SECONDS:
        raise RuntimeError("Projected time exceeds MAX_SECONDS; adjust grid or budget.")
    summary, paths = u.sweep(s, beta, N, RATIOS, ALPHAS, FIXED_LAMBDA, ACC_OBJECTIVE)
    summary.to_csv(run_dir/"summary.csv", index=False)
    paths.to_csv(run_dir/"paths.csv", index=False)
    log("DE sweep complete.")
(run_dir/"config.json").write_text(json.dumps(config, indent=2))
display(summary.head())
display(summary[summary.boundary == True][["policy","ratio","alpha"]])


In [ ]:
fig, axes = u.plot_policies(summary)
# 可自由修改：axes[0, 2].set_ylim(1e-10, 10)
fig.savefig(run_dir/"noise_policies.png", dpi=180, bbox_inches="tight")
plt.show()


## 每个噪声的完整 regularization path

不同 sigma 用不同颜色；实线 E_gen，虚线 E_acc。这里能看到优化点而不只是最终曲线。E_acc leading≈0 只是 ratio-of-moments cancellation，不保证真实 expected E_acc 为零。


In [ ]:
chosen_ratios = RATIOS[np.unique(np.linspace(0,len(RATIOS)-1,4).astype(int))]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ratio, color in zip(chosen_ratios, plt.get_cmap("viridis")(np.linspace(.05,.9,4))):
    part = paths[np.isclose(paths.ratio, ratio, rtol=1e-8, atol=0)].sort_values("alpha")
    label = rf"$\sigma={part.sigma.iloc[0]:.3g}$, $\sigma^2/S={ratio:g}$"
    axes[0].plot(part.lam, np.maximum(part.E_gen,1e-16), color=color, label=label)
    axes[0].plot(part.lam, np.maximum(part.E_acc,1e-16), "--", color=color)
    axes[1].plot(part.lam, part.slope_acc, color=color, label=label)
axes[0].set(xscale="log", yscale="log", xlabel="lambda", ylabel="E / S",
            title="Solid: E gen; dashed: E acc (leading)")
axes[1].set(xscale="log", xlabel="lambda", ylabel="accentuation slope")
axes[1].axhline(1, color=".5", ls=":")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=.2)
fig.tight_layout(); plt.show()


## 检查修正项与 near-zero minima

E_gen/S 是 noiseless teacher prediction MSE。E_acc/S=(1−N/D)²，而 R²_acc=1−(D/N−1)²，分母不同。E_acc_corrected 使用现有代码的 response-noise ratio mean/variance correction；它不是完整 design+noise Var(R)，也不是保证非负的精确风险。若修正后出现负值，不能把该数值当作真实 MSE 的改善。

In [ ]:
display(summary[["policy","sigma","ratio","alpha","lam","kappa","E_gen",
                 "E_acc","E_acc_corrected","R2_acc","boundary"]])
if (summary.E_acc_corrected < 0).any():
    print("Negative corrected approximation: inspect validity before optimizing it.")
# 检查更宽、更密网格：修改 ALPHAS 后重跑以上 cells。
# 修改绘图样式无需重新拟合：summary 和 paths 已缓存。
